# Train balance bot using PPO with domain randomization

Run each cell by pressing `shift + enter`.

We redo the original training and add more phases to our curriculum learning. In addition to training the bot to stay upright (phase 1) and reduce movement off the origin (phase 2), we introduce increasingly difficult domain randomization schemes in phases 3, 4 and 5.

In [1]:
# Import standard libraries
from dataclasses import replace
import os
from pathlib import Path
import sys
import time

# Third-party libraries
import gymnasium as gym
from gymnasium.wrappers import RecordEpisodeStatistics
import numpy as np
import torch

# Import custom environment
from balance_bot_env_cmd import BalanceBotEnv, DomainRandomConfig

# Add the folder containing our envs/ and rl/ packages to the path
sys.path.append("/workspace/software")

# Import PPO training module and exporter
from rl.ppo_trainer import PPOConfig, evaluate, train, export_tb_plots_as_csv, export_actor_onnx
from rl.onnx_actor_to_c import export_onnx_actor_to_c

In [2]:
# Settings
MJCF_PATH = Path("/workspace/mechanical/FreeCAD/bala2-fire/bala2-fire-simplified.xml")
SEED = 42
NUM_ENVS = 8               # Number of parallel environments. Only the first will be rendered.
STEPS_PER_ENV = 500_000    # Number of simulation steps to perform per environment

## Configure PPO and environment

In [3]:
# Configure PPO
ppo_config = PPOConfig(
    exp_name = "balance-bot-ppo",  # Name of the experiment
    env_id = "BalanceBot-v0",      # Name of the environment
    seed = SEED,                   # Constant seed for reproducibility
    num_envs = NUM_ENVS,           # Number of parallel environments
    actor_hidden_layers = 2,       # Number of hidden layers in the actor network
    actor_hidden_size = 32,        # Number of nodes in each hidden layer in the actor
    critic_hidden_layers = 2,      # Number of hidden layers in the critic network
    critic_hidden_size = 32,       # Number of nodes in each hidden layer in the critic
    total_timesteps = NUM_ENVS * STEPS_PER_ENV,  # Total simulation steps (all envs and iterations)
    num_steps = 2048,              # Number of steps per rollout per env (2048 * 0.002s = ~4 sec)
    num_minibatches = 32,          # Number of minibatches for each training epoch
    update_epochs = 10,            # Number of epochs to update actor and critic for each iteration
    anneal_lr = True,              # Enable annealing (lower learning rate as training goes on)
    learning_rate = 3e-4,          # Initial learning rate, reduced by annealing (if enabled)
    gamma = 0.99,                  # Discount factor (future rewards are discounted by this amount)
    gae_lambda = 0.95,             # GAE blending: 0 = pure TD error, 1 = pure Monte Carlo
    clip_coef = 0.2,               # Limits policy ratio to prevent large actor updates
    value_clip = 1.0,              # Absolute bounds on value prediction change per update (critic)
    ent_coef = 0.0,                # How much entropy factors into total loss calculation
    vf_coef = 0.5,                 # How much the value loss factors into total loss calculation
    max_grad_norm = 0.5,           # Limits how much actor/critic parameters can change during an update
    checkpoint_interval = 50,      # Save model every 50 iterations
    save_model = True,             # Save the final model
    timestep = 0.000,              # Match MJCF opt.timestep for real-time rendering (or 0 for fast)
)

In [4]:
def make_balance_bot_env(render, debug, **kwargs):
    """Function to create an environment for our balance bot"""
    # Create the environment and set the render mode
    env = BalanceBotEnv(
        mjcf_path = MJCF_PATH,
        render_mode = "human" if render else None,
        debug = debug,
        **kwargs
    )

    # Wrap in RecordEpisodeStatistics so we can log episodic returns in the 'info' dict
    return gym.wrappers.RecordEpisodeStatistics(env)

def make_envs(num_envs, **kwargs):
    """Create a SyncVectorEnv with num_envs balance bot environments."""
    env_factories = []
    for i in range(num_envs):
        env_factories.append(
            lambda render=(i==0), debug=(i==0), kw=kwargs: 
                make_balance_bot_env(render, debug=debug, **kw)
        )
    return gym.vector.SyncVectorEnv(env_factories)

In [5]:
def load_agent(envs, model_path):
    """
    For debugging only! Use this to load a previously trained model to skip previous phases. Note 
    that you will still need to run the cells in each prior phase that update the experiment name 
    and environment.
    """
    from rl.ppo_trainer import Agent, TrainResult

    # Make sure model_path is a Path
    model_path = Path(model_path)
    
    # Load agent from previous run
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    agent = Agent(envs, ppo_config).to(device)
    agent.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    agent.eval()
    print(f"Loaded model from {model_path}")
    
    # Wrap agent in a dummy result
    return TrainResult(
        agent = agent,
        checkpoint_dir = model_path.parent,
        best_model_path = model_path,
        final_model_path = None,
        best_mean_return = 0,
    )

## Phase 1: Balance only

We'll start with the easy task of only penalizing if the robot tilts (pitches) forward.

In [6]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-01"

# Create an environment that only rewards staying upright
envs = make_envs(
    NUM_ENVS,
    pitch_penalty_coef=0.5,
    action_penalty_coef=0.01,
    vel_reward_coef=0.0,
    yaw_reward_coef=0.0,
)

In [ ]:
# Choo choo train
result = train(ppo_config, envs=envs)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

In [ ]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

## Phase 2: Penalize position and rotation

Try to keep the robot in its original starting position and orientation.

In [7]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-02"

# Update the position and rotation coefficients in the existing environments
for env_stat_wrapper in envs.envs:
    env = env_stat_wrapper.env
    env.vel_reward_coef = 1.0
    env.yaw_reward_coef = 1.0
    env.sigma_vel = 1.0
    env.sigma_yaw = 0.5

In [ ]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

In [ ]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

## Phase 3: Small forward/back commands

In [10]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-03"

# Update domain randomization config
dr = DomainRandomConfig(
    cmd_vel_range=(-0.3, 0.3),  # Slow max forward/back commands
    cmd_yaw_range=(0.0, 0.0),   # No yaw commands yet
)

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env = env_stat_wrapper.env
    env.dr = dr
    env.sigma_vel = 0.1     # Small sigma: less reward for error
    env.sigma_yaw = 0.1     # Small sigma: less reward for error
    env.vel_max = 0.5       # Max speed in m/s
    env.yaw_max = 1.5       # Max rotational speed in rad/s

In [11]:
# DEBUG: Uncomment this cell to load a previously saved model
result = load_agent(envs, "runs/BalanceBot-v0__balance-bot-phase-02__42__1779582576/best_model.cleanrl_model")

Loaded model from runs/BalanceBot-v0__balance-bot-phase-02__42__1779582576/best_model.cleanrl_model


In [12]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-03__42__1779631708
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-03
New episode: cmd_vel=0.164, cmd_yaw=0.000
--- step 1000 ---
  cmd_vel=0.164, vel_target=0.082 m/s, vel_actual=-0.321 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=0.017 rad/s
  vel_tracking_reward=0.1970 (perfect=1.0)
  yaw_tracking_reward=0.9972 (perfect=1.0)
  pitch=-0.036 rad, pitch_penalty=0.0007
  pitch_rate=0.065 rad/s, pitch_rate_penalty=0.0000
  action_smoothness_penalty=0.0000
--- step 2000 ---
  cmd_vel=0.164, vel_target=0.082 m/s, vel_actual=-0.306 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=0.069 rad/s
  vel_tracking_reward=0.2212 (perfect=1.0)
  yaw_tracking_reward=0.9530 (perfect=1.0)
  pitch=-0.029 rad, pitch_penalty=0.0004
  pitch_rate=0.079 rad/s, pitch_rate_penalty=0.0000
  action_smoothness_penalty=0.0000
--- step 3000 ---
  cmd_vel=0.164, vel_target=0.082 m/s, vel_actual=-0.306 m/s
  cmd_yaw=0.000, yaw_targe

In [14]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

New episode: cmd_vel=0.259, cmd_yaw=0.000
--- step 1000 ---
  cmd_vel=0.259, vel_target=0.129 m/s, vel_actual=-0.046 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=0.007 rad/s
  vel_tracking_reward=0.7351 (perfect=1.0)
  yaw_tracking_reward=0.9996 (perfect=1.0)
  pitch=-0.035 rad, pitch_penalty=0.0006
  pitch_rate=-0.005 rad/s, pitch_rate_penalty=0.0000
  action_smoothness_penalty=0.0000
--- step 2000 ---
  cmd_vel=0.259, vel_target=0.129 m/s, vel_actual=-0.035 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=-0.014 rad/s
  vel_tracking_reward=0.7634 (perfect=1.0)
  yaw_tracking_reward=0.9981 (perfect=1.0)
  pitch=-0.033 rad, pitch_penalty=0.0006
  pitch_rate=-0.059 rad/s, pitch_rate_penalty=0.0000
  action_smoothness_penalty=0.0000
New episode: cmd_vel=-0.247, cmd_yaw=0.000
--- step 1000 ---
  cmd_vel=-0.247, vel_target=-0.124 m/s, vel_actual=-0.083 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=0.013 rad/s
  vel_tracking_reward=0.9834 (perfect=1.0)
  yaw_trac

## Phase 4: More forward/back speed, small yaw

In [ ]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-04"

# Update domain randomization config
dr.cmd_vel_range = (-0.6, 0.6)
dr.cmd_yaw_range = (-0.2, 0.2)

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env_stat_wrapper.env.dr = dr

In [ ]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

In [ ]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

## Phase 5: Full command range

In [ ]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-05"

# Update domain randomization config
dr.cmd_vel_range = (-1.0, 1.0)
dr.cmd_yaw_range = (-1.0, 1.0)

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env_stat_wrapper.env.dr = dr

In [ ]:
# DEBUG: Uncomment this cell to load a previously saved model
# result = load_agent(envs, "runs/BalanceBot-v0__balance-bot-phase-2__42__1778905085/best_model.cleanrl_model")

In [ ]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

In [ ]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

## Phase 6: Add observation noise and action delay DR

In [ ]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-06"

# Update domain randomization config
dr.pitch_noise_std_dev = 0.01
dr.pitch_rate_noise_std_dev = 0.01
dr.wheel_vel_noise_std_dev = 0.1
dr.action_delay_steps = 1
dr.action_delay_random = True

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env_stat_wrapper.env.dr = dr

In [ ]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

In [ ]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

## Phase 7: Add motor noise and pushes DR

In [ ]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-07"

# Update domain randomization config
dr.motor_noise_scale = 0.02
dr.push_prob = 0.005
dr.push_force_max_n = 0.3

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env_stat_wrapper.env.dr = dr

In [ ]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

In [ ]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

## Phase 8: Add mass and friction randomization

In [ ]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-08"

# Update domain randomization config
dr.mass_scale_range = (0.8, 1.2)
dr.friction_scale_range = (0.5, 1.5)

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env_stat_wrapper.env.dr = dr
    env = env_stat_wrapper.env
    env.vel_max = 2.0

In [ ]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

In [ ]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

## Phase 9: Add motor gain randomization

In [ ]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-09"

# Update domain randomization config
dr.motor_gain_range = (0.6, 1.0)

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env_stat_wrapper.env.dr = dr

In [ ]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

In [ ]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

## Phase 10: Random axle torques to simulate ridges

In [ ]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-10"

# Update domain randomization config
dr.ridge_prob = 0.05
dr.ridge_torque_max_nm = 0.005

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env_stat_wrapper.env.dr = dr

In [ ]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

In [ ]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

## Phase 11: Mid-episode sampling

In [ ]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-11"

# Add chance of chancing the command(s) mid-episode (every step)
dr.cmd_resample_prob = 0.005

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env_stat_wrapper.env.dr = dr
    env = env_stat_wrapper.env
    env.action_penalty_coef = 0.1 # TEST: penalize jittery actions more

In [ ]:
# DEBUG: Uncomment this cell to load a previously saved model
# result = load_agent(envs, "runs/BalanceBot-v0__balance-bot-phase-10__42__1779564068/best_model.cleanrl_model")

In [ ]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

In [ ]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

## Clean up and save model

At this point, we are done with training. We want to delete the environments and save the best actor from our final training phase. We'll export this actor network as an ONNX file that can be used on a variety of hardware platforms.

In [15]:
# Close the environments
for idx, env in enumerate(envs.envs):
    print(f"Closing env {idx}")
    env.env.close()

Closing env 0
Closing env 1
Closing env 2
Closing env 3
Closing env 4
Closing env 5
Closing env 6
Closing env 7


In [ ]:
# Get observation and action sizes
obs_size = envs.single_observation_space.shape[0]
action_size = envs.single_action_space.shape[0]

# Export the actor network as an ONNX model
export_actor_onnx(
    model_path=result.best_model_path,
    output_path=result.checkpoint_dir / "actor.onnx",
    obs_size=obs_size,
    action_size=action_size,
    num_hidden_layers=ppo_config.actor_hidden_layers,
    hidden_layer_size=ppo_config.actor_hidden_size,
)

In [ ]:
# Export actor to .h file
export_onnx_actor_to_c(
    onnx_path   = result.checkpoint_dir / "actor.onnx",
    output_path = result.checkpoint_dir / "actor.h"
)

## Debug

In [ ]:
# Create a single rendered eval env with fixed forward command
dr_test = DomainRandomConfig(
    cmd_vel_range=(-0.3, 0.3),
    cmd_yaw_range=(0.0, 0.0),
)
test_env = BalanceBotEnv(
    mjcf_path=MJCF_PATH,
    render_mode="human",
    debug=False,
    domain_rand=dr_test,
)
test_envs = gym.vector.SyncVectorEnv([lambda: test_env])

# Load your best model
result = load_agent(envs, "runs/BalanceBot-v0__balance-bot-phase-10__42__1779564068/best_model.cleanrl_model")
agent = result.agent
agent.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

obs, _ = test_envs.reset()
for i in range(2000):
    with torch.no_grad():
        action, _, _, _ = agent.get_action_and_value(torch.Tensor(obs).to(device))
    obs, reward, terminated, truncated, _ = test_envs.step(action.cpu().numpy())
    
    if i % 200 == 0:
        cvel = test_env.data.body("chassis").cvel
        print(f"step {i}:")
        print(f"  cvel={[f'{v:.3f}' for v in cvel]}")
        print(f"  qvel[6]={test_env.data.qvel[6]:.3f}, qvel[7]={test_env.data.qvel[7]:.3f}")
        print(f"  avg wheel vel={((test_env.data.qvel[6]+test_env.data.qvel[7])/2):.3f}")
    
    test_env.render()
    
    if terminated or truncated:
        print(f"Episode ended at step {i}")
        obs, _ = test_envs.reset()

test_envs.close()